# 15 — Adversarial Training and Robust Optimization — Lab

This lab gives you hands-on experience with adversarial training, the dominant empirical defense against adversarial examples. You implement the minimax robust optimization objective, build a PGD inner loop function, and train a model using PGD adversarial training. You then evaluate clean accuracy versus robust accuracy and compare your adversarially trained model to a standard-trained baseline model trained within the lab.

## Objectives

- Derive the minimax robust optimization problem and explain the role of the inner maximization and outer minimization loops.
- Implement standard adversarial training by augmenting minibatches with FGSM adversarial examples.
- Implement PGD adversarial training with specified step size, iteration count, and random restart.
- Train a PGD-adversarially trained model to convergence on CIFAR-10.
- Report and interpret the clean accuracy versus PGD-10 robust accuracy trade-off on a held-out test set.

## Prerequisites

You should be comfortable with the lecture material from this lesson, including the minimax robust optimization formulation, FGSM and PGD attacks, standard adversarial training, PGD adversarial training, and the accuracy-robustness trade-off. You should know what CIFAR-10 is, how to evaluate classification models with clean and robust accuracy, and have working knowledge of PyTorch's tensor operations, backward passes, and optimization loops.

## Required Software and Packages

- Python 3.10+
- PyTorch
- torchvision
- NumPy
- Matplotlib

No additional packages are required beyond the standard course environment.

## Environment Setup

Run the next cell to import the required libraries and set up the environment.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("Environment setup completed.")

## Background

Adversarial training solves a minimax robust optimization problem. The model learns to minimize the loss against an adversary's strongest attack:

min_theta max_{||delta||_p <= epsilon} L(f_theta(x + delta), y)

Standard adversarial training uses FGSM, which takes one step in the direction of the gradient. PGD adversarial training uses multiple gradient steps inside the maximization loop, making the inner attack much stronger. In this lab, you implement PGD adversarial training with k=7 steps, step size alpha=2/255, and epsilon=8/255 on CIFAR-10.

You will train a small convolutional neural network for 10 epochs using adversarial examples generated on each minibatch. You will then evaluate the model against PGD-10 attacks and compare the clean accuracy versus robust accuracy trade-off to a standard-trained baseline from m08.

## Background

Adversarial training solves a minimax robust optimization problem. The model learns to minimize the loss against an adversary's strongest attack:

min_theta max_{||delta||_p <= epsilon} L(f_theta(x + delta), y)

Standard adversarial training uses FGSM, which takes one step in the direction of the gradient. PGD adversarial training uses multiple gradient steps inside the maximization loop, making the inner attack much stronger. In this lab, you implement PGD adversarial training with k=7 steps, step size alpha=2/255, and epsilon=8/255 on CIFAR-10.

You will train a small convolutional neural network for 10 epochs using adversarial examples generated on each minibatch. You will also train a standard (non-adversarial) baseline model for comparison and evaluate both models against PGD-10 attacks to observe the accuracy-robustness trade-off.

## Exercise Instructions

1. Load the CIFAR-10 dataset and define a simple convolutional neural network (CNN) architecture. Print the model architecture and the shape of the first batch. (2 min)
2. Implement a PGD inner maximization function that takes a batch of images, the model, the true labels, and the attack parameters (epsilon, alpha, num_steps) and returns adversarial examples. (5 min)
3. Implement an adversarial training loop that trains the model on CIFAR-10 for 10 epochs using PGD adversarial training with the parameters: epsilon=8/255, alpha=2/255, num_steps=7, 1 random restart. Print training loss after each epoch. (6 min)
4. Train the model using the adversarial training loop and save the trained model to disk. (8 min)
5. Train a standard (non-adversarial) baseline model on CIFAR-10 for 10 epochs. Evaluate both the adversarially trained model and the standard baseline on clean accuracy and PGD-10 robust accuracy (epsilon=8/255). Report the results in a two-row table. (9 min)

In [ ]:
# Starter code: load CIFAR-10 and define a simple CNN.

# Data preprocessing and loading
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

# Simple CNN architecture
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Create model instance
model = SimpleCNN().to(device)
print("Model architecture:")
print(model)

# Get first batch to verify shapes
first_batch, first_labels = next(iter(train_loader))
first_batch = first_batch.to(device)
print(f"First batch shape: {first_batch.shape}")
print(f"First batch labels shape: {first_labels.shape}")

### Step 2: Implement PGD Inner Maximization

In [ ]:
# TODO: Implement PGD adversarial attack
def pgd_attack(model, x, y, epsilon, alpha, num_steps, device):
    """
    PGD attack: generate adversarial examples by maximizing the loss.
    
    Args:
        model: the neural network model
        x: input batch (requires_grad=True for gradient computation)
        y: target labels
        epsilon: perturbation budget (in range [0, 1] for normalized images)
        alpha: step size for PGD
        num_steps: number of PGD iterations
        device: torch device (cpu or cuda)
    
    Returns:
        x_adv: adversarial examples
    """
    # Initialize delta with random uniform noise within epsilon ball
    delta = torch.zeros_like(x, requires_grad=True)
    delta.data.uniform_(-epsilon, epsilon)
    
    # PGD loop: maximize loss for num_steps iterations
    for _ in range(num_steps):
        # Compute gradient with respect to input
        output = model(x + delta)
        loss = nn.CrossEntropyLoss()(output, y)
        
        # Compute gradient of loss w.r.t. delta
        grad = torch.autograd.grad(loss, delta, create_graph=False)[0]
        
        # Update delta in direction that increases loss
        with torch.no_grad():
            delta.data = delta.data + alpha * grad.sign()
            # Project back to epsilon ball
            delta.data = torch.clamp(delta.data, -epsilon, epsilon)
            # Clamp to valid image range [0, 1]
            delta.data = torch.clamp(x.data + delta.data, min=0, max=1) - x.data
    
    x_adv = x + delta
    return x_adv.detach()

# Test the PGD attack on a single batch
with torch.no_grad():
    test_x, test_y = next(iter(train_loader))
    test_x = test_x.to(device)
    test_y = test_y.to(device)

epsilon = 8.0 / 255.0
alpha = 2.0 / 255.0
num_steps = 7

x_adv = pgd_attack(model, test_x, test_y, epsilon, alpha, num_steps, device)
print(f"Original input shape: {test_x.shape}")
print(f"Adversarial input shape: {x_adv.shape}")
print(f"Max perturbation: {(x_adv - test_x).abs().max().item():.6f}")
print(f"Expected max perturbation (epsilon): {epsilon:.6f}")

### Step 3: Implement Adversarial Training Loop

In [ ]:
# TODO: Implement adversarial training loop
def adversarial_train_epoch(model, train_loader, optimizer, epsilon, alpha, num_steps, device):
    """
    Train the model for one epoch using adversarial training.
    
    Args:
        model: the neural network model
        train_loader: DataLoader for training data
        optimizer: optimizer (e.g., SGD or Adam)
        epsilon: perturbation budget
        alpha: PGD step size
        num_steps: number of PGD iterations
        device: torch device
    
    Returns:
        average loss over the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        # Generate adversarial examples using PGD
        x_adv = pgd_attack(model, x, y, epsilon, alpha, num_steps, device)
        
        # Train on adversarial examples
        optimizer.zero_grad()
        output = model(x_adv)
        loss = nn.CrossEntropyLoss()(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

# Create optimizer
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

# Adversarial training hyperparameters
epsilon = 8.0 / 255.0
alpha = 2.0 / 255.0
num_steps = 7

# Test one epoch
print("Testing adversarial training loop...")
avg_loss = adversarial_train_epoch(model, train_loader, optimizer, epsilon, alpha, num_steps, device)
print(f"Average training loss after first epoch test: {avg_loss:.4f}")

### Step 4: Train Model for 10 Epochs

In [ ]:
# TODO: Train for 10 epochs using adversarial training
num_epochs = 10
training_losses = []

print("Starting adversarial training for 10 epochs...")
for epoch in range(num_epochs):
    avg_loss = adversarial_train_epoch(model, train_loader, optimizer, epsilon, alpha, num_steps, device)
    training_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")

print("Training completed.")

# Save the adversarially trained model
torch.save(model.state_dict(), 'm15_adversarial_model.pt')
print("Model saved to m15_adversarial_model.pt")

### Step 5: Evaluate Clean and Robust Accuracy

In [ ]:
# TODO: Create and train a standard (non-adversarial) baseline model for comparison
# This model is trained on clean data only, with no adversarial augmentation.
# It serves as the "before" reference point in Step 5.

import copy

print("Training standard (non-adversarial) baseline model for comparison...")

# Create a fresh CNN model for the baseline
baseline_model = SimpleCNN().to(device)

# Define hyperparameters for baseline training
learning_rate = 0.01
criterion = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=learning_rate)

# Train on clean data only (no adversarial augmentation)
for epoch in range(num_epochs):
    baseline_model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        # Standard training: compute loss on clean examples only
        output = baseline_model(x)
        loss = criterion(output, y)
        
        baseline_optimizer.zero_grad()
        loss.backward()
        baseline_optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 2 == 0:
        print(f"Baseline epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("Baseline model training completed.")

# Save the baseline model
torch.save(baseline_model.state_dict(), 'm15_baseline_standard_model.pt')
print("Baseline model saved to m15_baseline_standard_model.pt")

In [ ]:
# TODO: Create and train a standard (non-adversarial) baseline model for comparison
# This model is trained on clean data only, with no adversarial augmentation.
# It serves as the "before" reference point in Step 5.

import copy

print("Training standard (non-adversarial) baseline model for comparison...")

# Create a fresh CNN model for the baseline
baseline_model = SimpleCNN().to(device)

# Define hyperparameters for baseline training
learning_rate = 0.01
criterion = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=learning_rate)

# Train on clean data only (no adversarial augmentation)
for epoch in range(num_epochs):
    baseline_model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        # Standard training: compute loss on clean examples only
        output = baseline_model(x)
        loss = criterion(output, y)
        
        baseline_optimizer.zero_grad()
        loss.backward()
        baseline_optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 2 == 0:
        print(f"Baseline epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("Baseline model training completed.")

# Save the baseline model
torch.save(baseline_model.state_dict(), 'm15_baseline_standard_model.pt')
print("Baseline model saved to m15_baseline_standard_model.pt")

### Expected Outputs

Step 1: The model summary prints the CNN architecture. The first batch shape is [128, 3, 32, 32] and labels shape is [128].

Step 2: The max perturbation should be approximately equal to epsilon (8/255 ≈ 0.0314).

Step 3: The test should print an average training loss in the range of 0.5 to 2.0.

Step 4: Training should complete in a reasonable time. The average loss per epoch should decrease over 10 epochs, starting from roughly 1.5 to 2.0 and reaching 0.3 to 0.8 by epoch 10.

Step 5: After training both the adversarially trained model and the standard baseline model for 10 epochs each, the comparison table should show that the adversarially trained model achieves roughly 60-70% clean accuracy and 50-60% robust accuracy (PGD-10), while the standard baseline achieves roughly 80-90% clean accuracy but only 5-15% robust accuracy (PGD-10). These numbers vary based on random initialization and training dynamics.

## Questions and Reflection

1. In the PGD inner maximization function, why do we initialize delta with random noise instead of zeros? How does this relate to the concept of random restarts mentioned in the lecture?

2. Why is the adversarially trained model's clean accuracy lower over the baseline standard-trained model? Explain how robust and non-robust features contribute to this trade-off based on the Ilyas et al. 2019 hypothesis from the lecture.

## Verification

In [ ]:
# Verification cells
print("VERIFICATION SUMMARY")
print("=" * 60)

# Check 1: Model is trained
print("Check 1: Model training completed")
assert len(training_losses) == 10, f"Expected 10 epochs, got {len(training_losses)}"
print("PASS: 10 epochs of training completed")

# Check 2: PGD attack produces valid perturbations
print("\nCheck 2: PGD attack produces valid perturbations")
test_x, test_y = next(iter(test_loader))
test_x = test_x.to(device)
test_y = test_y.to(device)
x_adv_test = pgd_attack(model, test_x, test_y, epsilon, alpha, num_steps, device)
max_pert = (x_adv_test - test_x).abs().max().item()
assert max_pert <= epsilon + 1e-5, f"Perturbation {max_pert} exceeds epsilon {epsilon}"
print(f"PASS: Max perturbation {max_pert:.6f} is within epsilon {epsilon:.6f}")

# Check 3: Accuracy values are in valid range
print("\nCheck 3: Accuracy values are in valid range")
assert 0 <= clean_acc_adv <= 100, f"Clean accuracy {clean_acc_adv} is out of range"
assert 0 <= robust_acc_adv <= 100, f"Robust accuracy {robust_acc_adv} is out of range"
print(f"PASS: Clean accuracy {clean_acc_adv:.2f}% and robust accuracy {robust_acc_adv:.2f}% are valid")

# Check 4: Robust accuracy is lower over clean accuracy (expected for adversarial training trade-off)
print("\nCheck 4: Robust accuracy is lower over clean accuracy")
print(f"PASS: Clean accuracy {clean_acc_adv:.2f}% > Robust accuracy {robust_acc_adv:.2f}%")

print("\n" + "=" * 60)
print("All verification checks PASSED")

## Optional Challenges

1. Implement standard FGSM adversarial training as an alternative to PGD. Train a second model using FGSM (only 1 step, same epsilon and alpha) and compare its clean and robust accuracy to the PGD-trained model. Which is stronger?

2. Experiment with different hyperparameters. Try training with alpha=1/255, num_steps=5, and num_steps=20 on separate models. How do these changes affect robust accuracy and training time?

## Completion Criteria

You have successfully completed this lab when:

1. You load CIFAR-10 data and define a simple CNN architecture, with the first batch shape verified.
2. You implement a pgd_attack function that generates valid adversarial examples within the epsilon ball.
3. You implement an adversarial_train_epoch function that trains on PGD adversarial examples generated on-the-fly.
4. You train the model for 10 epochs using PGD adversarial training, observing decreasing training loss across epochs.
5. You evaluate the model on clean accuracy and PGD-10 robust accuracy, producing numeric values for both.
6. The verification cell prints PASS for all four checks, confirming the implementation is correct and produces valid results.
7. You answer the two reflection questions about random restarts in PGD and the accuracy-robustness trade-off.